In [1]:
from google.colab import drive
import xarray as xr
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/fishing_project/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## STEP 1: Load and Subset Datasets

In [2]:
# West Philippine Sea bounding box
LAT_MIN, LAT_MAX = 10, 20
LON_MIN, LON_MAX = 114, 120
DATE_START = "2019-01-01"
DATE_END   = "2024-12-31"

# Physics (0.083° resolution)
physics_ds = xr.open_dataset(DATA_DIR + "cmems_mod_glo_phy_my_0.083deg_P1M-m_1777301394294.nc")

print(f"Available physics depths: {physics_ds.depth.values[:5]}")

# Select depth first with method='nearest', then slice spatial/temporal
physics_subset = physics_ds.sel(depth=0.49, method='nearest').sel(
    latitude=slice(LAT_MIN, LAT_MAX),
    longitude=slice(LON_MIN, LON_MAX),
    time=slice(DATE_START, DATE_END)
)

# BGC (0.25° resolution - this is our TARGET grid)
bgc_ds = xr.open_dataset(DATA_DIR + "cmems_mod_glo_bgc_my_0.25deg_P1M-m_1777301388002.nc")

print(f"Available BGC depths: {bgc_ds.depth.values[:5]}")

bgc_subset = bgc_ds.sel(
    depth=slice(0.51, 5.14),
    latitude=slice(LAT_MIN, LAT_MAX),
    longitude=slice(LON_MIN, LON_MAX),
    time=slice(DATE_START, DATE_END)
).mean(dim='depth')

print(f"\nPhysics (0.083°): {physics_subset.dims}")
print(f"BGC (0.25°): {bgc_subset.dims}")

Available physics depths: [0.494025 1.541375 2.645669 3.819495 5.078224]
Available BGC depths: [0.50576   1.5558553 2.6676817 3.8562799 5.1403613]

Physics (0.083°): FrozenMappingWarningOnValuesAccess({'time': 72, 'latitude': 121, 'longitude': 72})
BGC (0.25°): FrozenMappingWarningOnValuesAccess({'time': 72, 'latitude': 41, 'longitude': 25})


## STEP 2: Monthly Aggregation

In [3]:
# Monthly aggregation (if not already monthly)
physics_monthly = physics_subset.resample(time='1M').mean()
bgc_monthly = bgc_subset.resample(time='1M').mean()

print(f"Monthly physics: {physics_monthly.dims}")
print(f"Monthly BGC: {bgc_monthly.dims}")

/usr/local/lib/python3.12/dist-packages/xarray/groupers.py:530: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


Monthly physics: FrozenMappingWarningOnValuesAccess({'time': 72, 'latitude': 121, 'longitude': 72})
Monthly BGC: FrozenMappingWarningOnValuesAccess({'time': 72, 'latitude': 41, 'longitude': 25})


/usr/local/lib/python3.12/dist-packages/xarray/groupers.py:530: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  self.index_grouper = pd.Grouper(


## STEP 3: Define Target Grid and Regrid

In [4]:
# Use BGC's native coordinates as the standard 0.25° grid
target_lats = bgc_monthly.latitude.values
target_lons = bgc_monthly.longitude.values

print(f"Target grid: {len(target_lats)} × {len(target_lons)} at 0.25° resolution")

# Regrid physics to match BGC grid
physics_regrid = physics_monthly.interp(
    latitude=target_lats,
    longitude=target_lons,
    method='linear'
)

print(f"Physics regridded: {physics_regrid.dims}")

Target grid: 41 × 25 at 0.25° resolution
Physics regridded: FrozenMappingWarningOnValuesAccess({'time': 72, 'latitude': 41, 'longitude': 25})


## STEP 4: Verify Alignment

In [5]:
assert np.allclose(physics_regrid.latitude, bgc_monthly.latitude), "Latitude mismatch!"
assert np.allclose(physics_regrid.longitude, bgc_monthly.longitude), "Longitude mismatch!"
assert len(physics_regrid.time) == len(bgc_monthly.time), "Time mismatch!"

print("✔ All grids aligned to 0.25° resolution")

✔ All grids aligned to 0.25° resolution


## STEP 5: Gap Filling

In [6]:
def fill_gaps(data):
    """Fill NaN gaps with linear interpolation"""
    filled = data.interpolate_na(dim='time', method='linear', fill_value='extrapolate')
    return filled

physics_filled = fill_gaps(physics_regrid)
bgc_filled = fill_gaps(bgc_monthly)

print("✔ Gap filling complete")

✔ Gap filling complete


## STEP 6: Extract and Normalize Variables

In [7]:
def normalize(data, method='minmax'):
    """Normalize data using min-max or z-score"""
    if method == 'minmax':
        return (data - data.min()) / (data.max() - data.min())
    elif method == 'zscore':
        return (data - data.mean()) / data.std()
    return data

# Extract variables from correct datasets
# From BGC dataset (biogeochemical)
chl  = normalize(bgc_filled['chl'])   # Chlorophyll-a
nppv = normalize(bgc_filled['nppv'])  # Net primary production

# From Physics dataset
ssh = normalize(physics_filled['zos'])     # Sea surface height
sst = normalize(physics_filled['thetao'])  # Sea surface temperature
uo  = normalize(physics_filled['uo'])      # Eastward sea water velocity
vo  = normalize(physics_filled['vo'])      # Northward sea water velocity

print("✔ Normalization complete")
print(f"  Chl: {chl.shape}")
print(f"  NPP: {nppv.shape}")
print(f"  SSH: {ssh.shape}")
print(f"  SST: {sst.shape}")
print(f"  UO:  {uo.shape}")
print(f"  VO:  {vo.shape}")

✔ Normalization complete
  Chl: (72, 41, 25)
  NPP: (72, 41, 25)
  SSH: (72, 41, 25)
  SST: (72, 41, 25)
  UO:  (72, 41, 25)
  VO:  (72, 41, 25)


## STEP 7: Process AIS Data (Optional)

In [8]:
# STEP 7a: Load AIS CSVs (flat folder, bbox-filtered)
# Reads only 4 columns; skips files outside date range by filename;
# filters rows to bbox before concat — never loads global data into RAM.
import glob, os, time

AIS_ROOT = DATA_DIR + "ais_fishing/"
USE_COLS = ["date", "cell_ll_lat", "cell_ll_lon", "fishing_hours"]

def load_ais_filtered(ais_root, date_start, date_end,
                      lat_min, lat_max, lon_min, lon_max):
    start = pd.Timestamp(date_start)
    end   = pd.Timestamp(date_end)
    chunks = []

    all_files = sorted(glob.glob(os.path.join(ais_root, "*.csv")))
    print(f"Total CSVs in folder: {len(all_files)}")

    for fp in all_files:
        fname = os.path.basename(fp)
        # Filename ends: ...YYYY-MM-DD.csv  (last 14 chars before .csv)
        try:
            file_date = pd.Timestamp(fname[-14:-4])
        except Exception:
            continue
        if not (start <= file_date <= end):
            continue  # skip without opening

        for attempt in range(3):          # retry up to 3 times
            try:
                df = pd.read_csv(
                    fp,
                    usecols=USE_COLS,
                    dtype={
                        "cell_ll_lat":   "float32",
                        "cell_ll_lon":   "float32",
                        "fishing_hours": "float32",
                    }
                )
                mask = (
                    (df["cell_ll_lat"] >= lat_min) & (df["cell_ll_lat"] <  lat_max) &
                    (df["cell_ll_lon"] >= lon_min) & (df["cell_ll_lon"] <  lon_max)
                )
                filtered = df[mask]
                if not filtered.empty:
                    chunks.append(filtered)
                break                     # success — exit retry loop

            except OSError as e:
                if attempt < 2:
                    print(f"  Drive blip on {fname}, retrying ({attempt+1}/3)...")
                    time.sleep(5)         # wait 5s then retry
                else:
                    print(f"  Skipped {fname} after 3 attempts: {e}")

    if not chunks:
        raise ValueError("No AIS data found for the given bbox / date range!")

    ais_df = pd.concat(chunks, ignore_index=True)
    ais_df["date"]       = pd.to_datetime(ais_df["date"])
    ais_df["year_month"] = ais_df["date"].dt.to_period("M")
    return ais_df


print("Loading AIS CSVs...")
ais_df = load_ais_filtered(
    AIS_ROOT, DATE_START, DATE_END,
    LAT_MIN, LAT_MAX, LON_MIN, LON_MAX
)
print(f"AIS records in bbox : {len(ais_df):,}")
print(f"Date range          : {ais_df['date'].min().date()} to {ais_df['date'].max().date()}")
print(f"Unique months       : {ais_df['year_month'].nunique()}")


# STEP 7b: Aggregate 0.1° GFW rows → 0.25° monthly grid
def aggregate_ais_to_grid(ais_df, target_lats, target_lons):
    # GFW uses lower-left corners; +0.05 gets cell centre
    lat_c = ais_df["cell_ll_lat"].values + 0.05
    lon_c = ais_df["cell_ll_lon"].values + 0.05

    lat_idx = (np.searchsorted(target_lats, lat_c) - 1).clip(0, len(target_lats) - 1)
    lon_idx = (np.searchsorted(target_lons, lon_c) - 1).clip(0, len(target_lons) - 1)

    df = ais_df.copy()
    df["lat_idx"] = lat_idx
    df["lon_idx"] = lon_idx

    result = {}
    for period, grp in df.groupby("year_month"):
        grid = np.zeros((len(target_lats), len(target_lons)), dtype=np.float32)
        np.add.at(grid,
                  (grp["lat_idx"].values, grp["lon_idx"].values),
                  grp["fishing_hours"].values)
        result[str(period)] = grid
    return result


def ais_to_xarray(ais_grids, cmems_times, target_lats, target_lons):
    # Align monthly grids to the CMEMS time axis; missing months → 0
    data = np.zeros((len(cmems_times), len(target_lats), len(target_lons)), dtype=np.float32)
    for i, t in enumerate(cmems_times):
        key = str(pd.Timestamp(t).to_period("M"))
        if key in ais_grids:
            data[i] = ais_grids[key]
    return xr.DataArray(
        data,
        dims=["time", "latitude", "longitude"],
        coords={"time": cmems_times, "latitude": target_lats, "longitude": target_lons},
        name="fishing_hours"
    )


def normalize_ais(da):
    # Log1p then min-max — handles heavy-tailed fishing effort distribution
    log_da = np.log1p(da)
    mn, mx = float(log_da.min()), float(log_da.max())
    return (log_da - mn) / (mx - mn) if mx > mn else log_da * 0


ais_grids      = aggregate_ais_to_grid(ais_df, target_lats, target_lons)
ais_da         = ais_to_xarray(ais_grids, bgc_monthly.time.values, target_lats, target_lons)
ais_normalized = normalize_ais(ais_da)

print(f"Months aggregated : {len(ais_grids)}")
print(f"AIS DataArray     : {dict(ais_da.sizes)}")
print(f"AIS normalized    : min={float(ais_normalized.min()):.3f}  max={float(ais_normalized.max()):.3f}")

# Cache to Drive — skips reprocessing on Colab restart
ais_da.to_netcdf(DATA_DIR + "ais_fishing_effort_gridded.nc")
print("Saved: ais_fishing_effort_gridded.nc")


Loading AIS CSVs...
Total CSVs in folder: 72
AIS records in bbox : 141,381
Date range          : 2019-01-01 to 2024-12-01
Unique months       : 72
Months aggregated : 72
AIS DataArray     : {'time': 72, 'latitude': 41, 'longitude': 25}
AIS normalized    : min=0.000  max=1.000
Saved: ais_fishing_effort_gridded.nc


## STEP 8: Save Preprocessed Data

In [10]:
# Combine all features into one dataset (7 channels incl. AIS fishing effort)
preprocessed = xr.Dataset({
    'chl':            chl,
    'nppv':           nppv,
    'ssh':            ssh,
    'sst':            sst,
    'uo':             uo,             # Eastward sea water velocity
    'vo':             vo,             # Northward sea water velocity
    'fishing_effort': ais_normalized, # AIS apparent fishing hours (log-normalised)
})

# Save to NetCDF
preprocessed.to_netcdf(DATA_DIR + 'preprocessed_features.nc')

print("\nPreprocessed data saved -> preprocessed_features.nc")
print(f"Final dataset dimensions : {preprocessed.dims}")
print(f"Variables                : {list(preprocessed.data_vars)}")



Preprocessed data saved -> preprocessed_features.nc
Final dataset dimensions : FrozenMappingWarningOnValuesAccess({'latitude': 41, 'longitude': 25, 'time': 72})
Variables                : ['chl', 'nppv', 'ssh', 'sst', 'uo', 'vo', 'fishing_effort']
